# Topic 05 — Practice: GroupBy & Aggregation

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
oi = items.merge(orders[['order_id','channel','customer_id']], on='order_id', how='left')


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–04**:

1. (T04) How does standardising casing change Aurora's channel totals?
2. (T03) Why must boolean masks use `&` not `and`?
3. NumPy: how do you sum only the elements where a mask is True?

In [ ]:
# NumPy recall: segmented sum via masks (a manual 'groupby')
keys = np.array(['a','b','a','b','a']); vals = np.array([1.,2.,3.,4.,5.])
sum_a = ...   # TODO sum of vals where key=='a'
assert sum_a == 9
print(sum_a)

## 🎯 Core Lesson Tasks (current topic)

Split → apply → combine. Use named aggregation; `transform` to broadcast a group stat back to rows.

**Core 1 — channel revenue.** Total `line_revenue` per `channel` (use `oi`).

In [ ]:
channel_rev = ...
assert channel_rev.sum() > 0
channel_rev.sort_values(ascending=False)

**Core 2 — named aggregation.** Per `product_id`: `revenue` (sum), `units` (sum qty), `lines` (count).

In [ ]:
summary = ...
assert {'revenue','units','lines'} <= set(summary.columns)
summary.sort_values('revenue', ascending=False).head()

**Core 3 — transform.** Add `pct_of_product` = a line's revenue ÷ that product's total revenue.

In [ ]:
items['pct_of_product'] = ...
chk = items.groupby('product_id')['pct_of_product'].sum().dropna().mean()
assert np.isclose(chk, 1.0)
print('ok')

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 04) — clean then group.** Confirm `channel` has ≤4 distinct values before trusting the groupby.

In [ ]:
assert orders['channel'].nunique() <= 4
print(orders['channel'].unique())

**Mixed review (Topic 03) — filter + aggregate.** Revenue from *delivered* orders only.

In [ ]:
delivered_rev = ...
assert delivered_rev > 0
print(f'£{delivered_rev:,.0f}')

## 🕵️ Data Detective Investigation

**Case file #5 — top customer & AOV.** Management asks: who is the top customer by revenue, and what is the average order value (revenue per order)?

In [ ]:
order_rev = oi.groupby('order_id')['line_revenue'].sum()
aov = order_rev.mean()
cust_rev = oi.groupby('customer_id')['line_revenue'].sum()
top_customer = cust_rev.idxmax()
assert aov > 0
print('AOV £%.2f | top customer %s' % (aov, top_customer))

## 🔎 Interview Lens (write answers — no model answers)

- **Q14:** Explain split–apply–combine in your own words.
- **Q15:** Why choose `groupby` over `pivot_table` — and vice versa?
- **Q16:** Difference between `transform`, `agg`, and `apply` on a groupby.

## ✍️ Reflection (write short explanations)

1. Answer Q14 and Q16 in writing.
2. Did the channel revenue ranking match the *raw* (dirty) ranking? Why does cleaning matter here?
3. **Investigation log:** which channel/products drive revenue?